# ============================================================
# PharmaLens AI
# Notebook 08 - Drug Similarity Intelligence
# ============================================================
#
# Objective:
# Build an AI-driven pharmaceutical drug similarity engine
# using product, therapeutic, manufacturer, strength, price,
# market and sales information.
#
# Dataset:
# Egyptian Pharmaceutical Market
# Period: 2021-2025
#
# Main outputs:
# 1. Drug similarity profiles
# 2. Similarity scores
# 3. Top-N similar drugs
# 4. Competitive product mapping
# 5. Substitute / alternative drug identification
# 6. Similarity matrix
# 7. Strategic similarity insights
# ============================================================

In [1]:
# ============================================================
# PHARMALENS AI
# PROJECT 08 — DRUG SIMILARITY
# CELL 1 — LOAD DATA
# ============================================================

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ------------------------------------------------------------
# Processed data directory
# ------------------------------------------------------------

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("Project root:")
print(PROJECT_ROOT)

print("\nProcessed directory:")
print(PROCESSED_DIR)

# ------------------------------------------------------------
# Check directory
# ------------------------------------------------------------

if not PROCESSED_DIR.exists():
    raise FileNotFoundError(
        f"Processed directory not found:\n{PROCESSED_DIR}"
    )

# ------------------------------------------------------------
# Find parquet files
# ------------------------------------------------------------

parquet_files = sorted(
    PROCESSED_DIR.glob("*.parquet")
)

if not parquet_files:
    raise FileNotFoundError(
        f"No parquet files found in:\n{PROCESSED_DIR}"
    )

print("\nAvailable parquet files:")

for i, file in enumerate(parquet_files, 1):
    print(f"{i}. {file.name}")

# ------------------------------------------------------------
# Load first parquet file
# ------------------------------------------------------------

DATA_PATH = parquet_files[0]

print("\nLoading:")
print(DATA_PATH)

df = pd.read_parquet(DATA_PATH)

# Keep both names available
data = df.copy()

print("\n" + "=" * 60)
print("DATA LOADED SUCCESSFULLY")
print("=" * 60)

print(f"Rows: {len(data):,}")
print(f"Columns: {len(data.columns)}")

display(data.head())

Project root:
D:\learning\Epsilon\Data Science\PDF lec\Decodelab\PharmaLens AI

Processed directory:
D:\learning\Epsilon\Data Science\PDF lec\Decodelab\PharmaLens AI\data\processed

Available parquet files:
1. cleaned_pharma_data.parquet

Loading:
D:\learning\Epsilon\Data Science\PDF lec\Decodelab\PharmaLens AI\data\processed\cleaned_pharma_data.parquet

DATA LOADED SUCCESSFULLY
Rows: 909,922
Columns: 18


,Distribution Channel,Therapeutic Class,Manufacturer,Brand Name,Pack Size,Product Launch,Drug Strength,Selling Price,Market Category,Month,Year,Sales Units,Sales Value,Revenue_per_Unit,Launch_Year,Product_Age_Years,New_Product_Flag,Month_Number
0,PHARMACIES,A01A0 STOMATOLOGICALS,ADCO*,HEXITOL,MOUTH WASH 100ML,1990-04-01,000,49.9,KGA ORAL TOPICAL LIQUIDS,2021-01-01,2021,124790.0,1871850.0,15.0,1990.0,31.0,0,1
1,PHARMACIES,A01A0 STOMATOLOGICALS,ADCO*,HEXITOL,MOUTH WASH 100ML,1990-04-01,000,49.9,KGA ORAL TOPICAL LIQUIDS,2021-02-01,2021,112962.0,1694430.0,15.0,1990.0,31.0,0,2
2,PHARMACIES,A01A0 STOMATOLOGICALS,ADCO*,HEXITOL,MOUTH WASH 100ML,1990-04-01,000,49.9,KGA ORAL TOPICAL LIQUIDS,2021-03-01,2021,108014.0,1620210.0,15.0,1990.0,31.0,0,3
3,PHARMACIES,A01A0 STOMATOLOGICALS,ADCO*,HEXITOL,MOUTH WASH 100ML,1990-04-01,000,49.9,KGA ORAL TOPICAL LIQUIDS,2021-04-01,2021,88090.0,1321350.0,15.0,1990.0,31.0,0,4
4,PHARMACIES,A01A0 STOMATOLOGICALS,ADCO*,HEXITOL,MOUTH WASH 100ML,1990-04-01,000,49.9,KGA ORAL TOPICAL LIQUIDS,2021-05-01,2021,76111.0,1141665.0,15.0,1990.0,31.0,0,5


In [2]:
# ============================================================
# DATA VALIDATION
# ============================================================

EXPECTED_COLUMNS = [
    "Distribution Channel",
    "Therapeutic Class",
    "Manufacturer",
    "Brand Name",
    "Pack Size",
    "Product Launch",
    "Drug Strength",
    "Selling Price",
    "Market Category",
    "Month",
    "Year",
    "Sales Units",
    "Sales Value"
]

missing_columns = [
    col
    for col in EXPECTED_COLUMNS
    if col not in data.columns
]

if missing_columns:
    raise ValueError(
        "Missing required columns:\n"
        + "\n".join(
            f"- {col}"
            for col in missing_columns
        )
    )

print(
    "All required PharmaLens AI columns are available."
)

print(
    f"Total sales rows: {len(data):,}"
)

All required PharmaLens AI columns are available.
Total sales rows: 909,922


In [3]:
# ============================================================
# DATA TYPE CLEANING
# ============================================================

# ------------------------------------------------------------
# Numeric columns
# ------------------------------------------------------------

NUMERIC_COLUMNS = [
    "Selling Price",
    "Sales Units",
    "Sales Value",
    "Year"
]

for col in NUMERIC_COLUMNS:

    data[col] = pd.to_numeric(
        data[col],
        errors="coerce"
    )

# ------------------------------------------------------------
# Dates
# ------------------------------------------------------------

data["Product Launch"] = pd.to_datetime(
    data["Product Launch"],
    errors="coerce"
)

data["Month"] = pd.to_datetime(
    data["Month"],
    errors="coerce"
)

# ------------------------------------------------------------
# Text columns
# ------------------------------------------------------------

TEXT_COLUMNS = [
    "Distribution Channel",
    "Therapeutic Class",
    "Manufacturer",
    "Brand Name",
    "Pack Size",
    "Drug Strength",
    "Market Category"
]

for col in TEXT_COLUMNS:

    data[col] = (
        data[col]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
    )

print("Data types cleaned successfully.")

display(data.dtypes)

Data types cleaned successfully.


Distribution Channel            object
Therapeutic Class               object
Manufacturer                    object
Brand Name                      object
Pack Size                       object
Product Launch          datetime64[ns]
Drug Strength                   object
Selling Price                  float64
Market Category                 object
Month                   datetime64[ns]
Year                             int64
Sales Units                    float64
Sales Value                    float64
Revenue_per_Unit               float64
Launch_Year                    float64
Product_Age_Years              float64
New_Product_Flag                 int64
Month_Number                     int32
dtype: object

In [4]:
# ============================================================
# MASTER PRODUCT DATASET
# ============================================================

PRODUCT_KEYS = [
    "Brand Name",
    "Therapeutic Class",
    "Manufacturer",
    "Pack Size",
    "Drug Strength",
    "Market Category"
]

print("Creating product-level dataset...")

product_data = (
    data
    .groupby(
        PRODUCT_KEYS,
        dropna=False
    )
    .agg(
        Average_Price=(
            "Selling Price",
            "mean"
        ),

        Median_Price=(
            "Selling Price",
            "median"
        ),

        Total_Sales_Units=(
            "Sales Units",
            "sum"
        ),

        Total_Sales_Value=(
            "Sales Value",
            "sum"
        ),

        First_Launch_Date=(
            "Product Launch",
            "min"
        ),

        Last_Year=(
            "Year",
            "max"
        ),

        Number_of_Months=(
            "Month",
            "nunique"
        ),

        Distribution_Channels=(
            "Distribution Channel",
            "nunique"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# Product ID
# ------------------------------------------------------------

product_data.insert(
    0,
    "Product_ID",
    range(
        1,
        len(product_data) + 1
    )
)

# ------------------------------------------------------------
# Product age
# ------------------------------------------------------------

product_data["First_Launch_Date"] = pd.to_datetime(
    product_data["First_Launch_Date"],
    errors="coerce"
)

ANALYSIS_DATE = pd.Timestamp("2025-12-31")

product_data["Product_Age_Years"] = (
    (
        ANALYSIS_DATE
        - product_data["First_Launch_Date"]
    ).dt.days / 365.25
)

product_data["Product_Age_Years"] = (
    product_data["Product_Age_Years"]
    .clip(lower=0)
)

product_data["Product_Age_Years"] = (
    product_data["Product_Age_Years"]
    .fillna(
        product_data["Product_Age_Years"].median()
    )
)

print("=" * 60)
print("MASTER PRODUCT DATASET")
print("=" * 60)

print(f"Original sales rows : {len(data):,}")
print(f"Unique products     : {len(product_data):,}")

display(product_data.head())

Creating product-level dataset...
MASTER PRODUCT DATASET
Original sales rows : 909,922
Unique products     : 16,683


,Product_ID,Brand Name,Therapeutic Class,Manufacturer,Pack Size,Drug Strength,Market Category,Average_Price,Median_Price,Total_Sales_Units,Total_Sales_Value,First_Launch_Date,Last_Year,Number_of_Months,Distribution_Channels,Product_Age_Years
0,1,123,R05A0 COLD PREPARATIONS,HIKMA PLC*,C.TAB 10,000,ABA ORAL S ORD COATED TABLETS,3.0,3.0,293.00,8.790000e+02,2003-01-01,2023,1,1,22.997947
1,2,123,R05A0 COLD PREPARATIONS,HIKMA PLC*,C.TAB 20,000,ABA ORAL S ORD COATED TABLETS,40.0,40.0,18521747.89,4.152406e+08,2002-11-01,2025,60,2,23.164956
2,3,123,R05A0 COLD PREPARATIONS,HIKMA PLC*,SYRUP 120ML,000,DGM ORAL L ORD SYRUPS,32.0,32.0,41679950.37,7.824456e+08,2008-01-01,2025,60,2,17.998631
3,4,123 EXTRA,R05A0 COLD PREPARATIONS,HIKMA PLC*,FCT 650/60/ 4MG 20,4MG,ABC ORAL S ORD FILM-COATED TABS,50.0,50.0,1808244.50,8.545135e+07,2024-05-01,2025,20,2,1.667351
4,5,2 HC,A11G1 VITAMIN C PLAIN,NOVACARE*,FILM C.TABS 1000MG 20,1000MG,ABC ORAL S ORD FILM-COATED TABS,120.0,120.0,180.00,2.160000e+04,2024-12-01,2025,13,1,1.081451


In [5]:
# ============================================================
# PRODUCT DATA VALIDATION
# ============================================================

assert product_data["Product_ID"].is_unique
assert product_data.index.is_unique

required_product_columns = [
    "Product_ID",
    "Brand Name",
    "Therapeutic Class",
    "Manufacturer",
    "Pack Size",
    "Drug Strength",
    "Market Category",
    "Average_Price",
    "Total_Sales_Units",
    "Total_Sales_Value",
    "Product_Age_Years"
]

missing = [
    col
    for col in required_product_columns
    if col not in product_data.columns
]

assert not missing, (
    f"Missing product columns: {missing}"
)

print("Product dataset validation passed.")
print(
    f"Products available for similarity: "
    f"{len(product_data):,}"
)

Product dataset validation passed.
Products available for similarity: 16,683


In [21]:
# ============================================================
# SIMILARITY CONFIGURATION
# ============================================================

TOP_K = 20

TEXT_WEIGHT = 0.30
CATEGORICAL_WEIGHT = 0.25
NUMERICAL_WEIGHT = 0.15
PRICE_WEIGHT = 0.30

print("Similarity configuration:")
print(f"TOP_K = {TOP_K}")
print(f"Text weight = {TEXT_WEIGHT}")
print(f"Categorical weight = {CATEGORICAL_WEIGHT}")
print(f"Numerical weight = {NUMERICAL_WEIGHT}")
print(f"Price weight = {PRICE_WEIGHT}")

assert abs(
    (
        TEXT_WEIGHT
        + CATEGORICAL_WEIGHT
        + NUMERICAL_WEIGHT
        + PRICE_WEIGHT
    ) - 1.0
) < 0.0001

Similarity configuration:
TOP_K = 20
Text weight = 0.3
Categorical weight = 0.25
Numerical weight = 0.15
Price weight = 0.3


In [7]:
# ============================================================
# TEXT PROFILE
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer

TEXT_PROFILE_COLUMNS = [
    "Brand Name",
    "Therapeutic Class",
    "Manufacturer",
    "Pack Size",
    "Drug Strength",
    "Market Category"
]

text_data = (
    product_data[
        TEXT_PROFILE_COLUMNS
    ]
    .fillna("Unknown")
    .astype(str)
)

product_data["Drug_Text_Profile"] = (
    text_data["Brand Name"]
    + " "
    + text_data["Therapeutic Class"]
    + " "
    + text_data["Manufacturer"]
    + " "
    + text_data["Pack Size"]
    + " "
    + text_data["Drug Strength"]
    + " "
    + text_data["Market Category"]
)

print("Text profiles created.")

Text profiles created.


In [8]:
# ============================================================
# TEXT TOP-K SIMILARITY
# MEMORY SAFE
# ============================================================

from sklearn.neighbors import NearestNeighbors

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_features=50000,
    dtype=np.float32
)

tfidf_matrix = vectorizer.fit_transform(
    product_data["Drug_Text_Profile"]
)

print(
    "TF-IDF matrix:",
    tfidf_matrix.shape
)

text_nn = NearestNeighbors(
    n_neighbors=min(
        TOP_K + 1,
        len(product_data)
    ),
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

text_nn.fit(tfidf_matrix)

text_distances, text_indices = (
    text_nn.kneighbors(tfidf_matrix)
)

text_similarity_topk = {}

for i in range(
    len(product_data)
):

    matches = []

    for position in range(
        1,
        len(text_indices[i])
    ):

        j = int(
            text_indices[i][position]
        )

        similarity = (
            1.0
            - float(
                text_distances[i][position]
            )
        )

        matches.append(
            (
                j,
                max(
                    0.0,
                    similarity
                )
            )
        )

    text_similarity_topk[i] = matches

print(
    "Text Top-K similarity completed."
)

TF-IDF matrix: (16683, 24554)
Text Top-K similarity completed.


In [9]:
# ============================================================
# CATEGORICAL TOP-K SIMILARITY
# MEMORY SAFE
# ============================================================

CATEGORICAL_COLUMNS = [
    "Therapeutic Class",
    "Manufacturer",
    "Market Category",
    "Drug Strength",
    "Pack Size"
]

categorical_data = (
    product_data[
        CATEGORICAL_COLUMNS
    ]
    .fillna("Unknown")
    .astype(str)
    .apply(
        lambda col:
        col.str.strip().str.lower()
    )
)

# ------------------------------------------------------------
# Encode categories
# ------------------------------------------------------------

encoded = pd.DataFrame(
    index=product_data.index
)

for col in CATEGORICAL_COLUMNS:

    encoded[col] = (
        categorical_data[col]
        .astype("category")
        .cat.codes
        .astype(np.int32)
    )

# ------------------------------------------------------------
# Inverted index
# ------------------------------------------------------------

inverted_index = {}

for col in CATEGORICAL_COLUMNS:

    groups = (
        encoded
        .groupby(col)
        .groups
    )

    for value, indices in groups.items():

        inverted_index[
            (
                col,
                int(value)
            )
        ] = np.asarray(
            indices,
            dtype=np.int32
        )

# ------------------------------------------------------------
# Similarity
# ------------------------------------------------------------

categorical_similarity_topk = {}

n_products = len(product_data)

for i in range(n_products):

    candidate_counts = {}

    for col in CATEGORICAL_COLUMNS:

        value = int(
            encoded.iloc[i][col]
        )

        candidates = inverted_index.get(
            (
                col,
                value
            ),
            []
        )

        for j in candidates:

            if j == i:
                continue

            candidate_counts[j] = (
                candidate_counts.get(j, 0)
                + 1
            )

    similarities = [
        (
            j,
            shared / len(CATEGORICAL_COLUMNS)
        )
        for j, shared
        in candidate_counts.items()
    ]

    similarities.sort(
        key=lambda x: x[1],
        reverse=True
    )

    categorical_similarity_topk[i] = (
        similarities[:TOP_K]
    )

    if (
        (i + 1) % 2000 == 0
        or i == n_products - 1
    ):
        print(
            f"Processed "
            f"{i + 1:,}/{n_products:,}"
        )

print(
    "Categorical Top-K similarity completed."
)

Processed 2,000/16,683
Processed 4,000/16,683
Processed 6,000/16,683
Processed 8,000/16,683
Processed 10,000/16,683
Processed 12,000/16,683
Processed 14,000/16,683
Processed 16,000/16,683
Processed 16,683/16,683
Categorical Top-K similarity completed.


In [22]:
# ============================================================
# PRICE TOP-K SIMILARITY
# MEMORY SAFE
# ============================================================

from sklearn.neighbors import NearestNeighbors

prices = (
    product_data[
        ["Average_Price"]
    ]
    .fillna(0)
    .values
    .astype(np.float32)
)

price_features = np.log1p(
    np.clip(
        prices,
        0,
        None
    )
)

price_nn = NearestNeighbors(
    n_neighbors=min(
        TOP_K + 1,
        len(product_data)
    ),
    metric="euclidean",
    n_jobs=-1
)

price_nn.fit(
    price_features
)

price_distances, price_indices = (
    price_nn.kneighbors(
        price_features
    )
)

price_similarity_topk = {}

for i in range(
    len(product_data)
):

    matches = []

    price_a = (
        product_data.iloc[i][
            "Average_Price"
        ]
    )

    for position in range(
        1,
        len(price_indices[i])
    ):

        j = int(
            price_indices[i][position]
        )

        price_b = (
            product_data.iloc[j][
                "Average_Price"
            ]
        )

        if (
            pd.isna(price_a)
            or pd.isna(price_b)
        ):

            similarity = 0.0

        else:

            denominator = max(
                float(price_a),
                float(price_b),
                1e-9
            )

            similarity = (
                1.0
                -
                abs(
                    float(price_a)
                    -
                    float(price_b)
                )
                /
                denominator
            )

            similarity = float(
                np.clip(
                    similarity,
                    0.0,
                    1.0
                )
            )

        matches.append(
            (
                j,
                similarity
            )
        )

    matches.sort(
        key=lambda x: x[1],
        reverse=True
    )

    price_similarity_topk[i] = (
        matches[:TOP_K]
    )

print(
    "Price Top-K similarity completed."
)

Price Top-K similarity completed.


In [10]:
# ============================================================
# NUMERICAL TOP-K SIMILARITY
# MEMORY SAFE
# ============================================================

from sklearn.preprocessing import StandardScaler

NUMERICAL_COLUMNS = [
    "Average_Price",
    "Total_Sales_Units",
    "Total_Sales_Value",
    "Product_Age_Years",
    "Distribution_Channels"
]

numerical_features = (
    product_data[
        NUMERICAL_COLUMNS
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .copy()
)

numerical_features = (
    numerical_features
    .fillna(
        numerical_features.median()
    )
)

# ------------------------------------------------------------
# Log transform
# ------------------------------------------------------------

for col in [
    "Average_Price",
    "Total_Sales_Units",
    "Total_Sales_Value"
]:

    numerical_features[col] = np.log1p(
        numerical_features[col].clip(
            lower=0
        )
    )

# ------------------------------------------------------------
# Scale
# ------------------------------------------------------------

scaler = StandardScaler()

numerical_scaled = scaler.fit_transform(
    numerical_features
).astype(
    np.float32
)

# ------------------------------------------------------------
# Nearest neighbors
# ------------------------------------------------------------

numerical_nn = NearestNeighbors(
    n_neighbors=min(
        TOP_K + 1,
        len(product_data)
    ),
    metric="euclidean",
    n_jobs=-1
)

numerical_nn.fit(
    numerical_scaled
)

numerical_distances, numerical_indices = (
    numerical_nn.kneighbors(
        numerical_scaled
    )
)

# ------------------------------------------------------------
# Convert to similarity
# ------------------------------------------------------------

numerical_similarity_topk = {}

for i in range(
    len(product_data)
):

    matches = []

    for position in range(
        1,
        len(numerical_indices[i])
    ):

        j = int(
            numerical_indices[i][position]
        )

        distance = float(
            numerical_distances[i][position]
        )

        similarity = 1.0 / (
            1.0 + distance
        )

        matches.append(
            (
                j,
                similarity
            )
        )

    numerical_similarity_topk[i] = matches

print(
    "Numerical Top-K similarity completed."
)

Numerical Top-K similarity completed.


In [23]:
# ============================================================
# COMBINED SIMILARITY SCORE
# MEMORY SAFE
# ============================================================

combined_similarity_topk = {}

n_products = len(product_data)

print(
    f"Combining similarity scores "
    f"for {n_products:,} products..."
)

for i in range(n_products):

    candidates = set()

    # --------------------------------------------------------
    # Text
    # --------------------------------------------------------

    candidates.update(
        j
        for j, score
        in text_similarity_topk.get(
            i,
            []
        )
    )

    # --------------------------------------------------------
    # Categorical
    # --------------------------------------------------------

    candidates.update(
        j
        for j, score
        in categorical_similarity_topk.get(
            i,
            []
        )
    )

    # --------------------------------------------------------
    # Numerical
    # --------------------------------------------------------

    candidates.update(
        j
        for j, score
        in numerical_similarity_topk.get(
            i,
            []
        )
    )

    # --------------------------------------------------------
    # PRICE
    # --------------------------------------------------------

    candidates.update(
        j
        for j, score
        in price_similarity_topk.get(
            i,
            []
        )
    )

    text_scores = dict(
        text_similarity_topk.get(
            i,
            []
        )
    )

    categorical_scores = dict(
        categorical_similarity_topk.get(
            i,
            []
        )
    )

    numerical_scores = dict(
        numerical_similarity_topk.get(
            i,
            []
        )
    )

    price_scores = dict(
        price_similarity_topk.get(
            i,
            []
        )
    )

    results = []

    for j in candidates:

        text_score = (
            text_scores.get(
                j,
                0.0
            )
        )

        categorical_score = (
            categorical_scores.get(
                j,
                0.0
            )
        )

        numerical_score = (
            numerical_scores.get(
                j,
                0.0
            )
        )

        price_score = (
            price_scores.get(
                j,
                0.0
            )
        )

        final_score = (

            TEXT_WEIGHT
            * text_score

            +

            CATEGORICAL_WEIGHT
            * categorical_score

            +

            NUMERICAL_WEIGHT
            * numerical_score

            +

            PRICE_WEIGHT
            * price_score
        )

        results.append(
            (
                j,
                final_score,
                text_score,
                categorical_score,
                numerical_score,
                price_score
            )
        )

    results.sort(
        key=lambda x: x[1],
        reverse=True
    )

    combined_similarity_topk[i] = (
        results[:TOP_K]
    )

    if (
        (i + 1) % 2000 == 0
        or i == n_products - 1
    ):

        print(
            f"Processed "
            f"{i + 1:,}/{n_products:,}"
        )

print(
    "Combined similarity completed."
)

Combining similarity scores for 16,683 products...
Processed 2,000/16,683
Processed 4,000/16,683
Processed 6,000/16,683
Processed 8,000/16,683
Processed 10,000/16,683
Processed 12,000/16,683
Processed 14,000/16,683
Processed 16,000/16,683
Processed 16,683/16,683
Combined similarity completed.


In [24]:
# ============================================================
# SIMILARITY RESULTS TABLE
# ============================================================

records = []

for i, matches in (
    combined_similarity_topk.items()
):

    source = product_data.iloc[i]

    for rank, match in enumerate(
        matches,
        start=1
    ):

        (
            j,
            final_score,
            text_score,
            categorical_score,
            numerical_score,
            price_score
        ) = match

        target = product_data.iloc[j]

        original_price = (
            source["Average_Price"]
        )

        similar_price = (
            target["Average_Price"]
        )

        if (
            pd.isna(original_price)
            or pd.isna(similar_price)
            or original_price == 0
        ):

            price_difference = np.nan

        else:

            price_difference = (
                (
                    similar_price
                    -
                    original_price
                )
                /
                original_price
                * 100
            )

        records.append({

            "Product_ID":
                int(
                    source["Product_ID"]
                ),

            "Brand_Name":
                source["Brand Name"],

            "Manufacturer":
                source["Manufacturer"],

            "Therapeutic_Class":
                source["Therapeutic Class"],

            "Similar_Product_ID":
                int(
                    target["Product_ID"]
                ),

            "Similar_Brand_Name":
                target["Brand Name"],

            "Similar_Manufacturer":
                target["Manufacturer"],

            "Similar_Therapeutic_Class":
                target[
                    "Therapeutic Class"
                ],

            "Original_Price":
                original_price,

            "Similar_Price":
                similar_price,

            "Price_Difference_%":
                price_difference,

            "Similarity_Rank":
                rank,

            "Text_Similarity":
                round(
                    text_score,
                    4
                ),

            "Categorical_Similarity":
                round(
                    categorical_score,
                    4
                ),

            "Numerical_Similarity":
                round(
                    numerical_score,
                    4
                ),

            "Price_Similarity":
                round(
                    price_score,
                    4
                ),

            "Final_Similarity":
                round(
                    final_score,
                    4
                ),

            "Similarity_%":
                round(
                    final_score * 100,
                    2
                )
        })

similarity_results = pd.DataFrame(
    records
)

print(
    "Similarity results created."
)

print(
    f"Rows: {len(similarity_results):,}"
)

display(
    similarity_results.head(20)
)

Similarity results created.
Rows: 333,660


,Product_ID,Brand_Name,Manufacturer,Therapeutic_Class,Similar_Product_ID,Similar_Brand_Name,Similar_Manufacturer,Similar_Therapeutic_Class,Original_Price,Similar_Price,Price_Difference_%,Similarity_Rank,Text_Similarity,Categorical_Similarity,Numerical_Similarity,Price_Similarity,Final_Similarity,Similarity_%
0,1,123,HIKMA PLC*,R05A0 COLD PREPARATIONS,2,123,HIKMA PLC*,R05A0 COLD PREPARATIONS,3.0,40.00,1233.333333,1,0.8755,0.8,0.0000,0.0000,0.4626,46.26
1,1,123,HIKMA PLC*,R05A0 COLD PREPARATIONS,4308,DEXTROSE SALINE,HAIDYLENA*,K01B2 SOD CHLOR\CARBOHYD SOLNS,3.0,3.00,0.000000,2,0.0000,0.0,0.8429,1.0000,0.4264,42.64
2,1,123,HIKMA PLC*,R05A0 COLD PREPARATIONS,9233,MAGNESIUM SULPHATE,EIPICO/ACDIMA,A12C1 MAGNESIUM SUPPLEMENTS,3.0,3.00,0.000000,3,0.0000,0.0,0.6672,1.0000,0.4001,40.01
3,1,123,HIKMA PLC*,R05A0 COLD PREPARATIONS,3,123,HIKMA PLC*,R05A0 COLD PREPARATIONS,3.0,32.00,966.666667,4,0.5865,0.6,0.0000,0.0000,0.3260,32.60
4,1,123,HIKMA PLC*,R05A0 COLD PREPARATIONS,3695,COUGHSED-PARACETAM,ELERTE,R05A0 COLD PREPARATIONS,3.0,3.00,0.000000,5,0.0000,0.0,0.0000,1.0000,0.3000,30.00
5,1,123,HIKMA PLC*,R05A0 COLD PREPARATIONS,10151,MUCOVENT,MISR*,R05C0 EXPECTORANTS,3.0,2.95,-1.666667,6,0.0000,0.0,0.0000,0.9833,0.2950,29.50
6,1,123,HIKMA PLC*,R05A0 COLD PREPARATIONS,10960,OCUSOL,ALEXANDRIA*,S01A0 ANTI-INFECTIVES-EYE,3.0,3.10,3.333333,7,0.0000,0.0,0.0000,0.9677,0.2903,29.03
7,1,123,HIKMA PLC*,R05A0 COLD PREPARATIONS,10959,OCUSOL,ALEXANDRIA*,S01A0 ANTI-INFECTIVES-EYE,3.0,2.90,-3.333333,8,0.0000,0.0,0.0000,0.9667,0.2900,29.00
8,1,123,HIKMA PLC*,R05A0 COLD PREPARATIONS,4228,DEXAMETHA SOD PHOS,DELTA GRAND PH*,H02A1 INJ CORTICOSTEROIDS PLAIN,3.0,2.88,-4.000000,9,0.0000,0.0,0.0000,0.9600,0.2880,28.80
9,1,123,HIKMA PLC*,R05A0 COLD PREPARATIONS,13868,SINLERG,EVA PHARMA*,R05A0 COLD PREPARATIONS,3.0,56.00,1766.666667,10,0.4450,0.6,0.0000,0.0000,0.2835,28.35


In [13]:
# ============================================================
# SIMILARITY CLASSIFICATION
# ============================================================

def classify_similarity(score):

    if score >= 0.85:
        return "Very High"

    if score >= 0.70:
        return "High"

    if score >= 0.50:
        return "Moderate"

    if score >= 0.30:
        return "Low"

    return "Very Low"


similarity_results[
    "Similarity_Level"
] = (
    similarity_results[
        "Final_Similarity"
    ]
    .apply(classify_similarity)
)

display(
    similarity_results.head(20)
)

,Product_ID,Brand_Name,Manufacturer,Therapeutic_Class,Similar_Product_ID,Similar_Brand_Name,Similar_Manufacturer,Similar_Therapeutic_Class,Similarity_Rank,Text_Similarity,Categorical_Similarity,Numerical_Similarity,Final_Similarity,Similarity_%,Similarity_Level
0,1,123,HIKMA PLC*,R05A0 COLD PREPARATIONS,2,123,HIKMA PLC*,R05A0 COLD PREPARATIONS,1,0.8755,0.8,0.0000,0.6702,67.02,Moderate
1,1,123,HIKMA PLC*,R05A0 COLD PREPARATIONS,3,123,HIKMA PLC*,R05A0 COLD PREPARATIONS,2,0.5865,0.6,0.0000,0.4746,47.46,Low
2,1,123,HIKMA PLC*,R05A0 COLD PREPARATIONS,13868,SINLERG,EVA PHARMA*,R05A0 COLD PREPARATIONS,3,0.4450,0.6,0.0000,0.4180,41.80,Low
3,1,123,HIKMA PLC*,R05A0 COLD PREPARATIONS,13867,SINLERG,BIG PHARMA*,R05A0 COLD PREPARATIONS,4,0.4354,0.6,0.0000,0.4141,41.41,Low
4,1,123,HIKMA PLC*,R05A0 COLD PREPARATIONS,3516,COMTREX,HALEON*,R05A0 COLD PREPARATIONS,5,0.3877,0.6,0.0000,0.3951,39.51,Low
5,1,123,HIKMA PLC*,R05A0 COLD PREPARATIONS,12673,REDOX,INT.DRUG AGENCIES*,A11G1 VITAMIN C PLAIN,6,0.3627,0.6,0.0000,0.3851,38.51,Low
6,1,123,HIKMA PLC*,R05A0 COLD PREPARATIONS,4,123 EXTRA,HIKMA PLC*,R05A0 COLD PREPARATIONS,7,0.3935,0.4,0.0000,0.3174,31.74,Low
7,1,123,HIKMA PLC*,R05A0 COLD PREPARATIONS,4308,DEXTROSE SALINE,HAIDYLENA*,K01B2 SOD CHLOR\CARBOHYD SOLNS,8,0.0000,0.0,0.8429,0.1686,16.86,Very Low
8,1,123,HIKMA PLC*,R05A0 COLD PREPARATIONS,1288,ATSHI,AL-ROWAD PH*,R05A0 COLD PREPARATIONS,9,0.0000,0.4,0.0000,0.1600,16.00,Very Low
9,1,123,HIKMA PLC*,R05A0 COLD PREPARATIONS,1289,ATSHI,AL-ROWAD PH*,R05A0 COLD PREPARATIONS,10,0.0000,0.4,0.0000,0.1600,16.00,Very Low


In [14]:
# ============================================================
# FIND SIMILAR DRUGS
# ============================================================

def get_similar_drugs(
    brand_name,
    top_n=10
):
    """
    Find the most similar pharmaceutical products.
    """

    matches = similarity_results[
        similarity_results[
            "Brand_Name"
        ].astype(str).str.lower().str.strip()
        ==
        str(brand_name).lower().strip()
    ]

    return (
        matches
        .sort_values(
            "Final_Similarity",
            ascending=False
        )
        .head(top_n)
        .reset_index(drop=True)
    )


print(
    "get_similar_drugs() is ready."
)

get_similar_drugs() is ready.


In [25]:
# ============================================================
# PRICE ALTERNATIVES
# ============================================================

def get_price_alternatives(
    brand_name,
    top_n=10,
    max_price_difference_pct=30
):

    matches = similarity_results[
        similarity_results[
            "Brand_Name"
        ].astype(str)
        .str.lower()
        .str.strip()
        ==
        str(brand_name)
        .lower()
        .strip()
    ].copy()

    if matches.empty:
        return matches

    matches = matches[
        matches[
            "Price_Difference_%"
        ].abs()
        <= max_price_difference_pct
    ]

    return (
        matches
        .sort_values(
            [
                "Price_Similarity",
                "Final_Similarity"
            ],
            ascending=False
        )
        .head(top_n)
        .reset_index(drop=True)
    )


print(
    "get_price_alternatives() is ready."
)

get_price_alternatives() is ready.


In [26]:
# ============================================================
# LOWER-PRICE ALTERNATIVES
# ============================================================

def get_lower_price_alternatives(
    brand_name,
    top_n=10
):

    matches = similarity_results[
        similarity_results[
            "Brand_Name"
        ].astype(str)
        .str.lower()
        .str.strip()
        ==
        str(brand_name)
        .lower()
        .strip()
    ].copy()

    if matches.empty:
        return matches

    matches = matches[
        matches[
            "Similar_Price"
        ]
        <
        matches[
            "Original_Price"
        ]
    ]

    return (
        matches
        .sort_values(
            "Final_Similarity",
            ascending=False
        )
        .head(top_n)
        .reset_index(drop=True)
    )


print(
    "get_lower_price_alternatives() is ready."
)

get_lower_price_alternatives() is ready.


In [27]:
# ============================================================
# PREMIUM ALTERNATIVES
# ============================================================

def get_premium_alternatives(
    brand_name,
    top_n=10
):

    matches = similarity_results[
        similarity_results[
            "Brand_Name"
        ].astype(str)
        .str.lower()
        .str.strip()
        ==
        str(brand_name)
        .lower()
        .strip()
    ].copy()

    if matches.empty:
        return matches

    matches = matches[
        matches[
            "Similar_Price"
        ]
        >
        matches[
            "Original_Price"
        ]
    ]

    return (
        matches
        .sort_values(
            "Final_Similarity",
            ascending=False
        )
        .head(top_n)
        .reset_index(drop=True)
    )


print(
    "get_premium_alternatives() is ready."
)

get_premium_alternatives() is ready.


In [15]:
# ============================================================
# FIND COMPETITORS
# ============================================================

def get_competitors(
    brand_name,
    top_n=10
):
    """
    Find highly similar products
    from different manufacturers.
    """

    matches = similarity_results[
        similarity_results[
            "Brand_Name"
        ].astype(str).str.lower().str.strip()
        ==
        str(brand_name).lower().strip()
    ].copy()

    if matches.empty:
        return matches

    source_manufacturer = (
        matches.iloc[0][
            "Manufacturer"
        ]
    )

    matches = matches[
        matches[
            "Similar_Manufacturer"
        ]
        != source_manufacturer
    ]

    return (
        matches
        .sort_values(
            "Final_Similarity",
            ascending=False
        )
        .head(top_n)
        .reset_index(drop=True)
    )


print(
    "get_competitors() is ready."
)

get_competitors() is ready.


In [16]:
# ============================================================
# THERAPEUTIC ALTERNATIVES
# ============================================================

def get_therapeutic_alternatives(
    brand_name,
    top_n=10
):
    """
    Find similar drugs in the same therapeutic class
    from different manufacturers.
    """

    matches = similarity_results[
        similarity_results[
            "Brand_Name"
        ].astype(str).str.lower().str.strip()
        ==
        str(brand_name).lower().strip()
    ].copy()

    if matches.empty:
        return matches

    source_class = (
        matches.iloc[0][
            "Therapeutic_Class"
        ]
    )

    source_manufacturer = (
        matches.iloc[0][
            "Manufacturer"
        ]
    )

    matches = matches[
        (
            matches[
                "Similar_Therapeutic_Class"
            ]
            == source_class
        )
        &
        (
            matches[
                "Similar_Manufacturer"
            ]
            != source_manufacturer
        )
    ]

    return (
        matches
        .sort_values(
            "Final_Similarity",
            ascending=False
        )
        .head(top_n)
        .reset_index(drop=True)
    )


print(
    "get_therapeutic_alternatives() is ready."
)

get_therapeutic_alternatives() is ready.


In [17]:
# ============================================================
# COMPETITIVE PRESSURE
# ============================================================

competitive_pressure = (
    similarity_results[
        similarity_results[
            "Manufacturer"
        ]
        !=
        similarity_results[
            "Similar_Manufacturer"
        ]
    ]
    .groupby(
        [
            "Product_ID",
            "Brand_Name",
            "Manufacturer"
        ]
    )
    .agg(
        Number_of_Competitors=(
            "Similar_Product_ID",
            "nunique"
        ),

        Average_Competitor_Similarity=(
            "Final_Similarity",
            "mean"
        ),

        Maximum_Competitor_Similarity=(
            "Final_Similarity",
            "max"
        )
    )
    .reset_index()
)

competitive_pressure[
    "Competitive_Pressure_Score"
] = (
    competitive_pressure[
        "Average_Competitor_Similarity"
    ]
    *
    np.log1p(
        competitive_pressure[
            "Number_of_Competitors"
        ]
    )
)

competitive_pressure = (
    competitive_pressure
    .sort_values(
        "Competitive_Pressure_Score",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    competitive_pressure.head(20)
)

,Product_ID,Brand_Name,Manufacturer,Number_of_Competitors,Average_Competitor_Similarity,Maximum_Competitor_Similarity,Competitive_Pressure_Score
0,14390,SUBLINGAVIT,GLIMMAR PHARMA*,20,0.591195,0.8194,1.799906
1,6812,GLYSKIN,NICIM CO.*,20,0.581570,0.7026,1.770603
2,2626,CASTO,CASTLE PH.*,20,0.579120,0.8182,1.763144
3,13623,SENSICARE,ORIGINAL PH.*,20,0.568975,0.6680,1.732257
4,10463,NERVABROM,BROMED,20,0.567180,0.6728,1.726792
5,16049,VITAMERGE,TESLA,20,0.565695,0.6977,1.722271
6,15952,VINAPOTEX,AMC*,20,0.564840,0.6479,1.719668
7,842,ANGOSMOOTH,MASH*,20,0.561925,0.6878,1.710793
8,13193,ROWAMOXIFLOX,VORTEX,20,0.560595,0.6488,1.706744
9,14792,TEEMA,PICO,20,0.559930,0.6734,1.704719


In [28]:
# ============================================================
# PRICE COMPETITION ANALYSIS
# ============================================================

price_competition = (
    similarity_results
    .groupby(
        [
            "Product_ID",
            "Brand_Name",
            "Manufacturer"
        ]
    )
    .agg(

        Average_Similar_Product_Price=(
            "Similar_Price",
            "mean"
        ),

        Minimum_Similar_Product_Price=(
            "Similar_Price",
            "min"
        ),

        Maximum_Similar_Product_Price=(
            "Similar_Price",
            "max"
        ),

        Average_Price_Difference=(
            "Price_Difference_%",
            "mean"
        ),

        Number_of_Similar_Products=(
            "Similar_Product_ID",
            "nunique"
        )
    )
    .reset_index()
)

price_competition[
    "Price_Position"
] = np.select(

    [
        price_competition[
            "Average_Price_Difference"
        ] < -20,

        price_competition[
            "Average_Price_Difference"
        ] > 20
    ],

    [
        "Lower Priced",
        "Premium Priced"
    ],

    default="Market Aligned"
)

display(
    price_competition.head(20)
)

,Product_ID,Brand_Name,Manufacturer,Average_Similar_Product_Price,Minimum_Similar_Product_Price,Maximum_Similar_Product_Price,Average_Price_Difference,Number_of_Similar_Products,Price_Position
0,1,123,HIKMA PLC*,9.4815,2.75,56.0,216.050000,20,Premium Priced
1,2,123,HIKMA PLC*,36.2500,3.00,40.0,-9.375000,20,Market Aligned
2,3,123,HIKMA PLC*,28.5750,3.00,44.0,-10.703125,20,Market Aligned
3,4,123 EXTRA,HIKMA PLC*,50.0000,50.00,50.0,0.000000,20,Market Aligned
4,5,2 HC,NOVACARE*,121.0750,105.00,156.5,0.895833,20,Market Aligned
5,6,3 FLY,DRUG PHARMA EGYPT*,359.0000,140.00,390.0,-7.948718,20,Market Aligned
6,7,3 FLY,DRUG PHARMA EGYPT*,159.0000,140.00,390.0,13.571429,20,Market Aligned
7,8,3 FLY,DRUG PHARMA EGYPT*,195.0000,140.00,390.0,5.405405,20,Market Aligned
8,9,3 FLY,DRUG PHARMA EGYPT*,227.0000,140.00,390.0,0.888889,20,Market Aligned
9,10,3 WAY CALCIUM COMP,LIFE SERVICE*,49.6000,25.00,165.0,98.400000,20,Premium Priced


In [29]:
# ============================================================
# PRODUCT SIMILARITY PROFILE
# ============================================================

def get_similarity_profile(
    brand_name
):

    matches = similarity_results[
        similarity_results[
            "Brand_Name"
        ].astype(str)
        .str.lower()
        .str.strip()
        ==
        str(brand_name)
        .lower()
        .strip()
    ]

    if matches.empty:
        return {}

    top_match = (
        matches
        .sort_values(
            "Final_Similarity",
            ascending=False
        )
        .iloc[0]
    )

    return {

        "Brand":
            brand_name,

        "Original_Price":
            top_match[
                "Original_Price"
            ],

        "Closest_Product":
            top_match[
                "Similar_Brand_Name"
            ],

        "Overall_Similarity":
            top_match[
                "Final_Similarity"
            ],

        "Text_Similarity":
            top_match[
                "Text_Similarity"
            ],

        "Categorical_Similarity":
            top_match[
                "Categorical_Similarity"
            ],

        "Numerical_Similarity":
            top_match[
                "Numerical_Similarity"
            ],

        "Price_Similarity":
            top_match[
                "Price_Similarity"
            ],

        "Price_Difference_%":
            top_match[
                "Price_Difference_%"
            ]
    }


print(
    "get_similarity_profile() is ready."
)

get_similarity_profile() is ready.


In [18]:
# ============================================================
# BUSINESS INSIGHTS
# ============================================================

def generate_similarity_insights(
    brand_name,
    top_n=5
):
    """
    Generate business-oriented insights.
    """

    similar = get_similar_drugs(
        brand_name,
        top_n
    )

    if similar.empty:

        return {
            "Brand": brand_name,
            "Status": "Brand not found",
            "Insights": []
        }

    insights = []

    # --------------------------------------------------------
    # Closest product
    # --------------------------------------------------------

    top_match = similar.iloc[0]

    insights.append(
        f"Closest product: "
        f"{top_match['Similar_Brand_Name']} "
        f"with "
        f"{top_match['Similarity_%']:.1f}% "
        f"similarity."
    )

    # --------------------------------------------------------
    # Competitor
    # --------------------------------------------------------

    competitors = get_competitors(
        brand_name,
        top_n
    )

    if not competitors.empty:

        top_competitor = (
            competitors.iloc[0]
        )

        insights.append(
            f"Highest similarity competitor: "
            f"{top_competitor['Similar_Brand_Name']} "
            f"from "
            f"{top_competitor['Similar_Manufacturer']} "
            f"with "
            f"{top_competitor['Similarity_%']:.1f}% "
            f"similarity."
        )

    # --------------------------------------------------------
    # Alternatives
    # --------------------------------------------------------

    alternatives = (
        get_therapeutic_alternatives(
            brand_name,
            top_n
        )
    )

    if not alternatives.empty:

        insights.append(
            f"{len(alternatives)} "
            f"therapeutic alternatives identified."
        )

    return {
        "Brand": brand_name,
        "Status": "Success",
        "Insights": insights
    }


print(
    "Business insight function is ready."
)

Business insight function is ready.


In [33]:
# ============================================================
# PROJECT 08 — MARKET OPPORTUNITY
# ADD THIS CELL AT THE END OF THE NOTEBOOK
# ============================================================

import numpy as np
import pandas as pd

print("Calculating Market Opportunity...")

opportunity_df = data.copy()

# ------------------------------------------------------------
# Brand-level market metrics
# ------------------------------------------------------------

brand_metrics = (
    opportunity_df
    .groupby("Brand Name", dropna=False)
    .agg(
        Sales_Value=("Sales Value", "sum"),
        Sales_Units=("Sales Units", "sum"),
        Avg_Price=("Selling Price", "mean"),
        Manufacturer=("Manufacturer", "first"),
        Therapeutic_Class=("Therapeutic Class", "first"),
        Drug_Strength=("Drug Strength", "first"),
        Pack_Size=("Pack Size", "first"),
        Market_Category=("Market Category", "first")
    )
    .reset_index()
)

# ------------------------------------------------------------
# Growth calculation
# ------------------------------------------------------------

monthly_brand = (
    opportunity_df
    .groupby(
        ["Brand Name", "Year", "Month"],
        dropna=False
    )["Sales Value"]
    .sum()
    .reset_index()
)

monthly_brand["Period"] = pd.to_datetime(
    monthly_brand["Year"].astype(str)
    + "-"
    + monthly_brand["Month"].astype(str)
    + "-01",
    errors="coerce"
)

monthly_brand = monthly_brand.sort_values(
    ["Brand Name", "Period"]
)

monthly_brand["Previous_Sales"] = (
    monthly_brand
    .groupby("Brand Name")["Sales Value"]
    .shift(1)
)

monthly_brand["Growth_%"] = np.where(
    monthly_brand["Previous_Sales"] > 0,
    (
        (monthly_brand["Sales Value"]
         - monthly_brand["Previous_Sales"])
        / monthly_brand["Previous_Sales"]
    ) * 100,
    np.nan
)

growth_metrics = (
    monthly_brand
    .groupby("Brand Name")["Growth_%"]
    .mean()
    .reset_index()
    .rename(columns={"Growth_%": "Average_Growth_%"})
)

brand_metrics = brand_metrics.merge(
    growth_metrics,
    on="Brand Name",
    how="left"
)

# ------------------------------------------------------------
# Market share
# ------------------------------------------------------------

total_market_value = brand_metrics["Sales_Value"].sum()

brand_metrics["Market_Share_%"] = np.where(
    total_market_value > 0,
    brand_metrics["Sales_Value"]
    / total_market_value * 100,
    0
)

# ------------------------------------------------------------
# Opportunity score
# ------------------------------------------------------------

def percentile_score(series):
    if series.nunique(dropna=True) <= 1:
        return pd.Series(
            np.full(len(series), 50.0),
            index=series.index
        )

    return series.rank(
        pct=True,
        method="average"
    ) * 100


brand_metrics["Growth_Score"] = percentile_score(
    brand_metrics["Average_Growth_%"].fillna(0)
)

brand_metrics["Market_Share_Score"] = percentile_score(
    brand_metrics["Market_Share_%"].fillna(0)
)

brand_metrics["Revenue_Score"] = percentile_score(
    brand_metrics["Sales_Value"].fillna(0)
)

# ------------------------------------------------------------
# Opportunity Score
# ------------------------------------------------------------

brand_metrics["Market_Opportunity_Score"] = (
    brand_metrics["Growth_Score"] * 0.40
    + brand_metrics["Market_Share_Score"] * 0.30
    + brand_metrics["Revenue_Score"] * 0.30
)

# ------------------------------------------------------------
# Opportunity Tier
# ------------------------------------------------------------

brand_metrics["Opportunity_Tier"] = pd.cut(
    brand_metrics["Market_Opportunity_Score"],
    bins=[-np.inf, 25, 50, 75, np.inf],
    labels=[
        "Low Priority",
        "Monitor",
        "Strategic Opportunity",
        "High Priority"
    ]
)

print(
    "Market Opportunity calculated:",
    brand_metrics.shape
)

display(
    brand_metrics.sort_values(
        "Market_Opportunity_Score",
        ascending=False
    ).head(20)
)

Calculating Market Opportunity...


C:\Users\lapshop\AppData\Local\Temp\ipykernel_2428\1293638627.py:47: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  monthly_brand["Period"] = pd.to_datetime(


Market Opportunity calculated: (9958, 16)


,Brand Name,Sales_Value,Sales_Units,Avg_Price,Manufacturer,Therapeutic_Class,Drug_Strength,Pack_Size,Market_Category,Average_Growth_%,Market_Share_%,Growth_Score,Market_Share_Score,Revenue_Score,Market_Opportunity_Score,Opportunity_Tier
4466,INFLUVAC,9.911284e+08,3799372.11,322.000000,ABBOTT*,J07E1 INFLUENZA VACCINES,000,PREF.SYRINGE .5ML,FNA P ORD PRE-FILLED SYRINGES,656468.061617,0.112261,99.959831,97.951396,97.951396,98.754770,High Priority
1828,CIPRALEX,5.199028e+08,2306433.24,392.000000,LUNDBECK*,N06A4 SSRI ANTIDEPRESSANTS,10MG,FILM C.TABS 10MG 28,ABC ORAL S ORD FILM-COATED TABS,42111.032461,0.058887,98.885318,95.772243,95.772243,97.017473,High Priority
1347,CAL HEPARINE,8.010519e+08,8322720.91,167.609302,AMOUN PHARM.CO.*,B01B1 UNFRACTIONATED HEPARINS,12K,AMP. 12K 3 .5ML,FMA PARENTERAL ORDINARY AMPOULES,5657.190204,0.090732,96.113677,97.489456,97.489456,96.939144,High Priority
6190,NETLOOK,7.360823e+08,4135647.76,271.958702,AL ANDALOUS MEDICAL COMPANY*,D10B0 ORAL ANTI-ACNE PREPS,10MG,CAPS 10MG 20,ACA ORAL S ORD CAPSULES,4607.167414,0.083373,95.641695,97.188190,97.188190,96.569592,High Priority
6992,PEPTIC CARE,4.915787e+08,3517208.83,224.461538,MULTICARE*,A02B2 PROTON PUMP INHIBITORS,000,ENTER C.TABS 28,ABD ORAL S ORD ENTERIC-CTD TABS,12109.050182,0.055679,97.630046,95.501105,95.501105,96.352681,High Priority
5902,MOTILIUM,6.687950e+08,10830824.71,98.572034,JOHNSON & JOHNSON*,A03F0 GASTROPROKINETICS,10MG,FILM C.TABS 10MG 40,ABC ORAL S ORD FILM-COATED TABS,3825.772214,0.075751,95.169713,96.896967,96.896967,96.206065,High Priority
8757,TEBONINA,3.995195e+08,6907398.59,90.000000,DR WILLMAR SCHWABE,C04A1 CEREB/PERIPH VASOTHERAPS,000,FILM C.TABS FORT 20,ABC ORAL S ORD FILM-COATED TABS,19104.618917,0.045252,98.282788,94.637477,94.637477,96.095602,High Priority
3632,FUCICORT,1.230864e+09,30556694.83,76.561605,LEO*,D07B1 WITH ANTIBACTERIALS,000,CREAM 15G,MTA TOPICAL EXTERNAL CREAMS,1830.047508,0.139414,92.458325,98.393252,98.393252,96.019281,High Priority
5762,MINALAX,4.085987e+08,33989136.32,18.000000,AMOUN PHARM.CO.*,A06A2 STIMULANT LAXATIVES,000,ENTER.C.TABS 10,ABD ORAL S ORD ENTERIC-CTD TABS,15365.438052,0.046280,97.941354,94.717815,94.717815,96.007230,High Priority
4223,HI-POTENCY,3.135941e+08,3012607.35,150.000000,NERHADOU INTERNAT.,A11X9 ALL OTHER VITAMINS,000,TABS 30,AAA ORAL S ORD TABLETS,515354.978838,0.035519,99.929705,93.382205,93.382205,96.001205,High Priority


In [34]:
# ============================================================
# PRICE SIMILARITY
# ============================================================

print("Calculating price similarity features...")

price_df = (
    data.groupby("Brand Name", dropna=False)
    .agg(
        Avg_Price=("Selling Price", "mean"),
        Min_Price=("Selling Price", "min"),
        Max_Price=("Selling Price", "max"),
        Median_Price=("Selling Price", "median")
    )
    .reset_index()
)

# Avoid division by zero
price_df["Price_Range_%"] = np.where(
    price_df["Avg_Price"] > 0,
    (
        (price_df["Max_Price"] - price_df["Min_Price"])
        / price_df["Avg_Price"]
    ) * 100,
    0
)

# ------------------------------------------------------------
# Price similarity function
# ------------------------------------------------------------

def calculate_price_similarity(
    target_price,
    candidate_prices
):
    """
    Returns price similarity from 0 to 1.

    1.0 = identical price
    0.0 = very different price
    """

    candidate_prices = np.asarray(
        candidate_prices,
        dtype=float
    )

    if target_price <= 0:
        return np.zeros(len(candidate_prices))

    difference = np.abs(
        candidate_prices - target_price
    ) / target_price

    similarity = 1 / (1 + difference)

    return similarity


print("Price similarity features created.")

Calculating price similarity features...
Price similarity features created.


In [35]:
# ============================================================
# FINAL DRUG INTELLIGENCE
# ============================================================

drug_intelligence_df = brand_metrics.merge(
    price_df,
    on="Brand Name",
    how="left"
)

# ------------------------------------------------------------
# Drug Intelligence Score
# ------------------------------------------------------------

drug_intelligence_df["Drug_Intelligence_Score"] = (
    drug_intelligence_df["Market_Opportunity_Score"] * 0.60
    + drug_intelligence_df["Market_Share_Score"] * 0.20
    + drug_intelligence_df["Growth_Score"] * 0.20
)

# ------------------------------------------------------------
# Strategic classification
# ------------------------------------------------------------

def classify_drug(row):

    score = row["Drug_Intelligence_Score"]
    growth = row["Average_Growth_%"]
    share = row["Market_Share_%"]

    if score >= 75 and growth > 0:
        return "High Potential"

    elif score >= 60:
        return "Growth Opportunity"

    elif share >= 5 and growth <= 0:
        return "Defend / Mature"

    elif score >= 40:
        return "Monitor"

    else:
        return "Low Priority"


drug_intelligence_df["Drug_Strategic_Class"] = (
    drug_intelligence_df.apply(
        classify_drug,
        axis=1
    )
)

# ------------------------------------------------------------
# Final ranking
# ------------------------------------------------------------

drug_intelligence_df["Drug_Rank"] = (
    drug_intelligence_df[
        "Drug_Intelligence_Score"
    ]
    .rank(
        ascending=False,
        method="dense"
    )
)

drug_intelligence_df = (
    drug_intelligence_df
    .sort_values(
        "Drug_Intelligence_Score",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "Final Drug Intelligence created:",
    drug_intelligence_df.shape
)

display(
    drug_intelligence_df.head(20)
)

Final Drug Intelligence created: (9958, 24)


,Brand Name,Sales_Value,Sales_Units,Avg_Price_x,Manufacturer,Therapeutic_Class,Drug_Strength,Pack_Size,Market_Category,Average_Growth_%,...,Market_Opportunity_Score,Opportunity_Tier,Avg_Price_y,Min_Price,Max_Price,Median_Price,Price_Range_%,Drug_Intelligence_Score,Drug_Strategic_Class,Drug_Rank
0,INFLUVAC,9.911284e+08,3799372.11,322.000000,ABBOTT*,J07E1 INFLUENZA VACCINES,000,PREF.SYRINGE .5ML,FNA P ORD PRE-FILLED SYRINGES,656468.061617,...,98.754770,High Priority,322.000000,322.00,322.0,322.0,0.000000,98.835107,High Potential,1.0
1,CIPRALEX,5.199028e+08,2306433.24,392.000000,LUNDBECK*,N06A4 SSRI ANTIDEPRESSANTS,10MG,FILM C.TABS 10MG 28,ABC ORAL S ORD FILM-COATED TABS,42111.032461,...,97.017473,High Priority,392.000000,392.00,392.0,392.0,0.000000,97.141996,High Potential,2.0
2,CAL HEPARINE,8.010519e+08,8322720.91,167.609302,AMOUN PHARM.CO.*,B01B1 UNFRACTIONATED HEPARINS,12K,AMP. 12K 3 .5ML,FMA PARENTERAL ORDINARY AMPOULES,5657.190204,...,96.939144,High Priority,167.609302,16.50,198.0,198.0,108.287546,96.884113,High Potential,3.0
3,NETLOOK,7.360823e+08,4135647.76,271.958702,AL ANDALOUS MEDICAL COMPANY*,D10B0 ORAL ANTI-ACNE PREPS,10MG,CAPS 10MG 20,ACA ORAL S ORD CAPSULES,4607.167414,...,96.569592,High Priority,271.958702,150.00,394.0,274.0,89.719505,96.507732,High Potential,4.0
4,PEPTIC CARE,4.915787e+08,3517208.83,224.461538,MULTICARE*,A02B2 PROTON PUMP INHIBITORS,000,ENTER C.TABS 28,ABD ORAL S ORD ENTERIC-CTD TABS,12109.050182,...,96.352681,High Priority,224.461538,170.00,230.0,230.0,26.730637,96.437839,High Potential,5.0
5,HI-POTENCY,3.135941e+08,3012607.35,150.000000,NERHADOU INTERNAT.,A11X9 ALL OTHER VITAMINS,000,TABS 30,AAA ORAL S ORD TABLETS,515354.978838,...,96.001205,High Priority,150.000000,150.00,150.0,150.0,0.000000,96.263105,High Potential,6.0
6,TEBONINA,3.995195e+08,6907398.59,90.000000,DR WILLMAR SCHWABE,C04A1 CEREB/PERIPH VASOTHERAPS,000,FILM C.TABS FORT 20,ABC ORAL S ORD FILM-COATED TABS,19104.618917,...,96.095602,High Priority,90.000000,90.00,90.0,90.0,0.000000,96.241414,High Potential,7.0
7,DIGESTIN,3.333021e+08,15904629.51,42.000000,PHARCO*,A09A0 DIGESTIVES INCL ENZYMES,000,TABS 20,AAA ORAL S ORD TABLETS,106497.134052,...,95.975095,High Priority,42.000000,42.00,42.0,42.0,0.000000,96.203254,High Potential,8.0
8,MUCOTEC,3.230216e+08,4679866.08,79.324921,GLOBAL NAPI*,R05C0 EXPECTORANTS,150MG,CAPS 150MG 10,ACA ORAL S ORD CAPSULES,143114.817711,...,95.946977,High Priority,79.324921,11.00,144.0,56.0,167.664837,96.190400,High Potential,9.0
9,BIOVIT-12,3.712919e+08,26649810.82,28.000000,MUP/ACDIMA,B03X0 OTH ANTI-ANAEM+FOLIC ACID,000,A.IM 2 2ML,GMD PARENT RET I M AMPOULES,23223.569946,...,95.975095,High Priority,28.000000,28.00,28.0,28.0,0.000000,96.143001,High Potential,10.0


In [36]:
# ============================================================
# MEMORY-SAFE SIMILAR DRUG SEARCH
# ============================================================

def find_similar_drugs(
    brand_name,
    top_n=10
):
    """
    Find similar drugs without creating a huge
    NxN similarity matrix.
    """

    target = data[
        data["Brand Name"].astype(str).str.lower()
        == str(brand_name).lower()
    ]

    if target.empty:
        return pd.DataFrame(
            columns=[
                "Brand Name",
                "Manufacturer",
                "Therapeutic Class",
                "Drug Strength",
                "Pack Size",
                "Selling Price",
                "Similarity Score"
            ]
        )

    target_row = target.iloc[0]

    candidates = data[
        data["Brand Name"].astype(str).str.lower()
        != str(brand_name).lower()
    ].copy()

    if candidates.empty:
        return pd.DataFrame()

    # --------------------------------------------------------
    # Therapeutic similarity
    # --------------------------------------------------------

    candidates["Therapeutic_Similarity"] = (
        candidates["Therapeutic Class"].fillna("").astype(str)
        ==
        str(target_row["Therapeutic Class"])
    ).astype(float)

    # --------------------------------------------------------
    # Manufacturer similarity
    # --------------------------------------------------------

    candidates["Manufacturer_Similarity"] = (
        candidates["Manufacturer"].fillna("").astype(str)
        ==
        str(target_row["Manufacturer"])
    ).astype(float)

    # --------------------------------------------------------
    # Drug strength similarity
    # --------------------------------------------------------

    candidates["Strength_Similarity"] = (
        candidates["Drug Strength"].fillna("").astype(str)
        ==
        str(target_row["Drug Strength"])
    ).astype(float)

    # --------------------------------------------------------
    # Pack size similarity
    # --------------------------------------------------------

    candidates["Pack_Similarity"] = (
        candidates["Pack Size"].fillna("").astype(str)
        ==
        str(target_row["Pack Size"])
    ).astype(float)

    # --------------------------------------------------------
    # Price similarity
    # --------------------------------------------------------

    target_price = pd.to_numeric(
        pd.Series([target_row["Selling Price"]]),
        errors="coerce"
    ).iloc[0]

    candidate_price = pd.to_numeric(
        candidates["Selling Price"],
        errors="coerce"
    )

    if pd.notna(target_price) and target_price > 0:

        candidates["Price_Similarity"] = (
            1
            /
            (
                1
                +
                (
                    abs(candidate_price - target_price)
                    / target_price
                )
            )
        ).fillna(0)

    else:
        candidates["Price_Similarity"] = 0

    # --------------------------------------------------------
    # Combined score
    # --------------------------------------------------------

    candidates["Similarity Score"] = (
        candidates["Therapeutic_Similarity"] * 0.35
        + candidates["Strength_Similarity"] * 0.20
        + candidates["Pack_Similarity"] * 0.10
        + candidates["Price_Similarity"] * 0.25
        + candidates["Manufacturer_Similarity"] * 0.10
    )

    # --------------------------------------------------------
    # Aggregate by brand
    # --------------------------------------------------------

    result = (
        candidates
        .sort_values(
            "Similarity Score",
            ascending=False
        )
        .groupby("Brand Name", as_index=False)
        .first()
    )

    result = result[
        [
            "Brand Name",
            "Manufacturer",
            "Therapeutic Class",
            "Drug Strength",
            "Pack Size",
            "Selling Price",
            "Similarity Score"
        ]
    ]

    return result.head(top_n).reset_index(drop=True)


print("find_similar_drugs() is ready.")

find_similar_drugs() is ready.


In [39]:
# ============================================================
# DRUG INTELLIGENCE FUNCTION — FINAL SAFE VERSION
# ============================================================

def drug_intelligence(brand_name, top_n=10):
    """
    Complete intelligence profile for a drug/brand.

    Includes:
    - Brand information
    - Sales
    - Market share
    - Growth
    - Average price
    - Market opportunity
    - Strategic classification
    - Similar drugs
    """

    matches = drug_intelligence_df[
        drug_intelligence_df["Brand Name"]
        .astype(str)
        .str.lower()
        ==
        str(brand_name).lower()
    ]

    if matches.empty:
        return {
            "status": "not_found",
            "brand_name": brand_name,
            "message": "Brand not found in dataset."
        }

    drug = matches.iloc[0]

    # --------------------------------------------------------
    # SAFE PRICE EXTRACTION
    # --------------------------------------------------------

    if "Avg_Price" in drug.index:
        average_price = drug["Avg_Price"]

    elif "Average_Price" in drug.index:
        average_price = drug["Average_Price"]

    elif "Selling Price" in drug.index:
        average_price = drug["Selling Price"]

    else:
        average_price = (
            pd.to_numeric(
                data.loc[
                    data["Brand Name"].astype(str).str.lower()
                    == str(brand_name).lower(),
                    "Selling Price"
                ],
                errors="coerce"
            ).mean()
        )

    # --------------------------------------------------------
    # SAFE FIELD EXTRACTION
    # --------------------------------------------------------

    def get_value(column, default=None):
        if column in drug.index:
            return drug[column]
        return default

    # --------------------------------------------------------
    # SIMILAR DRUGS
    # --------------------------------------------------------

    similar = find_similar_drugs(
        brand_name,
        top_n=top_n
    )

    # --------------------------------------------------------
    # FINAL INTELLIGENCE RESULT
    # --------------------------------------------------------

    return {
        "status": "success",

        "brand_name": get_value(
            "Brand Name",
            brand_name
        ),

        "manufacturer": get_value(
            "Manufacturer"
        ),

        "therapeutic_class": get_value(
            "Therapeutic_Class",
            get_value("Therapeutic Class")
        ),

        "drug_strength": get_value(
            "Drug_Strength",
            get_value("Drug Strength")
        ),

        "pack_size": get_value(
            "Pack_Size",
            get_value("Pack Size")
        ),

        "market_category": get_value(
            "Market_Category",
            get_value("Market Category")
        ),

        "average_price": average_price,

        "sales_value": get_value(
            "Sales_Value",
            0
        ),

        "sales_units": get_value(
            "Sales_Units",
            0
        ),

        "market_share": get_value(
            "Market_Share_%",
            0
        ),

        "growth": get_value(
            "Average_Growth_%",
            0
        ),

        "market_opportunity_score": get_value(
            "Market_Opportunity_Score",
            0
        ),

        "opportunity_tier": get_value(
            "Opportunity_Tier",
            "Not Classified"
        ),

        "drug_intelligence_score": get_value(
            "Drug_Intelligence_Score",
            0
        ),

        "strategic_class": get_value(
            "Drug_Strategic_Class",
            "Not Classified"
        ),

        "similar_drugs": similar.to_dict(
            orient="records"
        )
    }


print("drug_intelligence() is ready.")

drug_intelligence() is ready.


In [30]:
# ============================================================
# EXPORT PROJECT 08 RESULTS
# ============================================================

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Main similarity results
# ------------------------------------------------------------

similarity_path = (
    OUTPUT_DIR
    / "drug_similarity_results.parquet"
)

similarity_results.to_parquet(
    similarity_path,
    index=False
)

# ------------------------------------------------------------
# Product dataset
# ------------------------------------------------------------

product_path = (
    OUTPUT_DIR
    / "drug_similarity_products.parquet"
)

product_data.to_parquet(
    product_path,
    index=False
)

# ------------------------------------------------------------
# Price competition
# ------------------------------------------------------------

price_path = (
    OUTPUT_DIR
    / "drug_price_competition.parquet"
)

price_competition.to_parquet(
    price_path,
    index=False
)

print("=" * 60)
print("PROJECT 08 EXPORT COMPLETED")
print("=" * 60)

print(
    f"Similarity results:\n{similarity_path}"
)

print(
    f"\nProduct data:\n{product_path}"
)

print(
    f"\nPrice competition:\n{price_path}"
)

PROJECT 08 EXPORT COMPLETED
Similarity results:
D:\learning\Epsilon\Data Science\PDF lec\Decodelab\PharmaLens AI\data\processed\drug_similarity_results.parquet

Product data:
D:\learning\Epsilon\Data Science\PDF lec\Decodelab\PharmaLens AI\data\processed\drug_similarity_products.parquet

Price competition:
D:\learning\Epsilon\Data Science\PDF lec\Decodelab\PharmaLens AI\data\processed\drug_price_competition.parquet


In [40]:
# ============================================================
# FINAL PROJECT 08 TEST — SAFE VERSION
# ============================================================

test_brand = (
    data["Brand Name"]
    .dropna()
    .astype(str)
    .iloc[0]
)

result = drug_intelligence(
    test_brand,
    top_n=5
)

print("=" * 60)
print("PROJECT 08 — DRUG INTELLIGENCE TEST")
print("=" * 60)

print("Brand:", result["brand_name"])
print("Manufacturer:", result["manufacturer"])
print("Therapeutic Class:", result["therapeutic_class"])
print("Drug Strength:", result["drug_strength"])
print("Pack Size:", result["pack_size"])
print("Average Price:", result["average_price"])
print("Sales Value:", result["sales_value"])
print("Sales Units:", result["sales_units"])
print("Market Share:", result["market_share"])
print("Growth:", result["growth"])
print(
    "Market Opportunity Score:",
    result["market_opportunity_score"]
)
print(
    "Opportunity Tier:",
    result["opportunity_tier"]
)
print(
    "Drug Intelligence Score:",
    result["drug_intelligence_score"]
)
print(
    "Strategic Class:",
    result["strategic_class"]
)

print("\nSimilar Drugs:")

if result["similar_drugs"]:
    display(
        pd.DataFrame(
            result["similar_drugs"]
        )
    )
else:
    print("No similar drugs found.")

print("\nProject 08 test completed successfully.")

PROJECT 08 — DRUG INTELLIGENCE TEST
Brand: HEXITOL
Manufacturer: ADCO*
Therapeutic Class: A01A0 STOMATOLOGICALS
Drug Strength: 000
Pack Size: MOUTH WASH 100ML
Average Price: 49.89999999999997
Sales Value: 168440788.4
Sales Units: 6391652.05
Market Share: 0.019078527432127533
Growth: 15688.87809945906
Market Opportunity Score: 92.72946374774051
Opportunity Tier: High Priority
Drug Intelligence Score: 93.08093994778068
Strategic Class: High Potential

Similar Drugs:


,Brand Name,Manufacturer,Therapeutic Class,Drug Strength,Pack Size,Selling Price,Similarity Score
0,123,HIKMA PLC*,R05A0 COLD PREPARATIONS,000,C.TAB 20,40.0,0.408612
1,123 EXTRA,HIKMA PLC*,R05A0 COLD PREPARATIONS,4MG,FCT 650/60/ 4MG 20,50.0,0.249500
2,2 HC,NOVACARE*,A11G1 VITAMIN C PLAIN,1000MG,FILM C.TABS 1000MG 20,120.0,0.103958
3,3 FLY,DRUG PHARMA EGYPT*,N07X0 ALL OTHER CNS DRUGS,300MG,FILM C.TABS 300MG 20,140.0,0.089107
4,3 WAY CALCIUM COMP,LIFE SERVICE*,A12A0 CALCIUM,000,CAPS 30,25.0,0.366778



Project 08 test completed successfully.


PROJECT 08 — DRUG SIMILARITY
│
├── Raw Data
│
├── Text Similarity
│
├── Categorical Similarity
│
├── Numerical Similarity
│
├── Combined Technical Similarity
│
├── Price Similarity
│
├── Pharma-Aware Similarity          ← ADD
│
├── Competitive Relevance Score      ← ADD
│
├── YoY Growth                       ← FIX
│
├── Market Opportunity               ← RECALCULATE
│
├── Drug Intelligence Score          ← RECALCULATE
│
├── Similar Drug Search              ← OVERRIDE
│
└── Final Drug Intelligence          ← OVERRIDE

In [ ]:
#################################################################################################

                    PRODUCT
                       │
        ┌──────────────┼──────────────┐
        ↓              ↓              ↓
     TEXT          CATEGORICAL     NUMERICAL
        │              │              │
        │              │        ┌─────┼─────┐
        │              │        ↓     ↓     ↓
        │              │      PRICE  SALES  AGE
        │              │
        └──────────────┼──────────────┘
                       ↓
              FEATURE SIMILARITY
                       ↓
              COMBINED SCORE
                       ↓
        ┌──────────────┼──────────────┐
        ↓              ↓              ↓
   Similar Drugs   Competitors   Alternatives
        ↓              ↓              ↓
      Price         Competitive    Therapeutic
   Alternatives      Pressure      Alternatives

In [ ]:
###################################################################################################

In [ ]:
################################################################################################3

In [ ]:
########################################################################3

In [ ]:
####################################################

In [ ]:
##################################################################

In [ ]:
##################################################

In [ ]:
################################################################

In [ ]:
###############################################################

In [ ]:
###########################################################################

In [ ]:
###########################################################

#Delete next codes

In [26]:
# ============================================================
# SIMILARITY WEIGHTS
# ============================================================

TEXT_WEIGHT = 0.35
THERAPEUTIC_WEIGHT = 0.30
NUMERICAL_WEIGHT = 0.15
PRODUCT_ATTRIBUTE_WEIGHT = 0.20

print("Similarity weights:")
print("Text:", TEXT_WEIGHT)
print("Therapeutic/Product attributes:", THERAPEUTIC_WEIGHT)
print("Numerical:", NUMERICAL_WEIGHT)
print("Product attributes:", PRODUCT_ATTRIBUTE_WEIGHT)

print(
    "\nTotal:",
    TEXT_WEIGHT +
    THERAPEUTIC_WEIGHT +
    NUMERICAL_WEIGHT +
    PRODUCT_ATTRIBUTE_WEIGHT
)

Similarity weights:
Text: 0.35
Therapeutic/Product attributes: 0.3
Numerical: 0.15
Product attributes: 0.2

Total: 1.0


In [19]:
# ============================================================
# TEST SIMILARITY ENGINE
# ============================================================

example_brand = product_df[
    "Brand Name"
].iloc[0]

print("Target drug:", example_brand)

similar_drugs = find_similar_drugs(
    example_brand,
    top_n=10
)

similar_drugs

Target drug: 123


NameError: name 'find_similar_drugs' is not defined

In [ ]:
# ============================================================
# PRICE SIMILARITY TEST
# ============================================================

price_results = price_similarity(
    example_brand,
    top_n=10
)

price_results

In [ ]:
# ============================================================
# HIGH-SIMILARITY COMPETITIVE NETWORK
# ============================================================

high_similarity_pairs = (
    top_similarity_df[
        top_similarity_df["Similarity_Score"] >= 0.70
    ]
    .copy()
)

high_similarity_pairs = (
    high_similarity_pairs
    .sort_values(
        "Similarity_Score",
        ascending=False
    )
)

print(
    "High similarity relationships:",
    len(high_similarity_pairs)
)

high_similarity_pairs.head(20)

In [ ]:
# ============================================================
# SIMILARITY + MARKET OPPORTUNITY
# ============================================================

opportunity_df = (
    top_similarity_df
    .merge(
        sales_ranking[
            [
                "Product_ID",
                "Total_Sales_Value",
                "Sales_Rank"
            ]
        ],
        on="Product_ID",
        how="left"
    )
    .merge(
        sales_ranking[
            [
                "Product_ID",
                "Total_Sales_Value"
            ]
        ].rename(
            columns={
                "Product_ID":
                    "Similar_Product_ID",
                "Total_Sales_Value":
                    "Similar_Total_Sales_Value"
            }
        ),
        on="Similar_Product_ID",
        how="left"
    )
)

opportunity_df.head()

In [ ]:
# ============================================================
# HIGH-VALUE SIMILAR PRODUCT OPPORTUNITIES
# ============================================================

high_value_similarity = opportunity_df[
    (
        opportunity_df["Similarity_Score"] >= 0.70
    )
    &
    (
        opportunity_df["Similar_Total_Sales_Value"]
        > opportunity_df["Similar_Total_Sales_Value"]
        .quantile(0.75)
    )
].copy()

high_value_similarity = (
    high_value_similarity
    .sort_values(
        [
            "Similarity_Score",
            "Similar_Total_Sales_Value"
        ],
        ascending=False
    )
)

high_value_similarity.head(20)

In [ ]:
# ============================================================
# HIGH-VALUE SIMILAR PRODUCT OPPORTUNITIES
# ============================================================

high_value_similarity = opportunity_df[
    (
        opportunity_df["Similarity_Score"] >= 0.70
    )
    &
    (
        opportunity_df["Similar_Total_Sales_Value"]
        > opportunity_df["Similar_Total_Sales_Value"]
        .quantile(0.75)
    )
].copy()

high_value_similarity = (
    high_value_similarity
    .sort_values(
        [
            "Similarity_Score",
            "Similar_Total_Sales_Value"
        ],
        ascending=False
    )
)

high_value_similarity.head(20)

In [ ]:
# ============================================================
# FINAL DRUG INTELLIGENCE
# ============================================================

drug_intelligence = product_df.copy()

drug_intelligence = drug_intelligence.merge(
    competitor_pressure[
        [
            "Product_ID",
            "Average_Competitor_Similarity",
            "Number_of_Similar_Competitors",
            "Max_Competitor_Similarity",
            "Competitive_Pressure_Score"
        ]
    ],
    on="Product_ID",
    how="left"
)

drug_intelligence[
    [
        "Average_Competitor_Similarity",
        "Max_Competitor_Similarity",
        "Competitive_Pressure_Score"
    ]
] = drug_intelligence[
    [
        "Average_Competitor_Similarity",
        "Max_Competitor_Similarity",
        "Competitive_Pressure_Score"
    ]
].fillna(0)

drug_intelligence.head()

In [ ]:
# ============================================================
# STRATEGIC DRUG CLASSIFICATION
# ============================================================

def strategic_similarity_category(row):
    
    pressure = row[
        "Competitive_Pressure_Score"
    ]
    
    sales = row[
        "Total_Sales_Value"
    ]
    
    sales_median = drug_intelligence[
        "Total_Sales_Value"
    ].median()
    
    if (
        pressure >=
        drug_intelligence[
            "Competitive_Pressure_Score"
        ].quantile(0.75)
        and
        sales >= sales_median
    ):
        return "High-Value High-Competition"
    
    elif (
        pressure >=
        drug_intelligence[
            "Competitive_Pressure_Score"
        ].quantile(0.75)
    ):
        return "High Competitive Pressure"
    
    elif sales >= sales_median:
        return "Strong Market Position"
    
    else:
        return "Low Competitive Pressure"


drug_intelligence[
    "Strategic_Similarity_Category"
] = drug_intelligence.apply(
    strategic_similarity_category,
    axis=1
)

drug_intelligence[
    "Strategic_Similarity_Category"
].value_counts()

In [ ]:
# ============================================================
# OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = "../data/drug_similarity_outputs"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

print("Output directory:", OUTPUT_DIR)

In [ ]:
# ============================================================
# EXPORT PRODUCT INTELLIGENCE
# ============================================================

drug_intelligence.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "drug_intelligence.csv"
    ),
    index=False
)

print("Saved: drug_intelligence.csv")

In [ ]:
# ============================================================
# EXPORT TOP SIMILARITY RESULTS
# ============================================================

top_similarity_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "top_drug_similarity.csv"
    ),
    index=False
)

print("Saved: top_drug_similarity.csv")

In [ ]:
# ============================================================
# EXPORT COMPETITIVE OPPORTUNITIES
# ============================================================

competitive_opportunities.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "competitive_drug_opportunities.csv"
    ),
    index=False
)

print("Saved: competitive_drug_opportunities.csv")

In [ ]:
# ============================================================
# EXPORT SIMILARITY PAIRS
# ============================================================

similarity_pairs.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "drug_similarity_pairs.csv"
    ),
    index=False
)

print("Saved: drug_similarity_pairs.csv")

In [ ]:
# ============================================================
# EXPORT SIMILARITY MATRIX
# ============================================================

similarity_matrix.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "drug_similarity_matrix.csv"
    )
)

print("Saved: drug_similarity_matrix.csv")

In [ ]:
# ============================================================
# EXECUTIVE SUMMARY
# ============================================================

print("=" * 70)
print("PHARMALENS AI - DRUG SIMILARITY INTELLIGENCE")
print("=" * 70)

print(f"\nTotal unique products: {len(product_df):,}")

print(
    f"High similarity relationships: "
    f"{len(high_similarity_pairs):,}"
)

print(
    f"Average similarity score: "
    f"{upper_triangle.mean():.3f}"
)

print(
    f"Maximum similarity score: "
    f"{upper_triangle.max():.3f}"
)

print(
    f"Products with competitive similarity: "
    f"{competitive_opportunities['Product_ID'].nunique():,}"
)

print("\nStrategic categories:")

print(
    drug_intelligence[
        "Strategic_Similarity_Category"
    ].value_counts()
)

print("\nTop competitive opportunities:")

display(
    competitive_opportunities[
        [
            "Brand_Name",
            "Manufacturer",
            "Similar_Brand_Name",
            "Similar_Manufacturer",
            "Similarity_%",
            "Similar_Sales_Value"
        ]
    ].head(10)
)

print("\n" + "=" * 70)
print("DRUG SIMILARITY ANALYSIS COMPLETED")
print("=" * 70)

In [ ]:
# ============================================================
# EXECUTIVE SUMMARY
# ============================================================

print("=" * 70)
print("PHARMALENS AI - DRUG SIMILARITY INTELLIGENCE")
print("=" * 70)

print(f"\nTotal unique products: {len(product_df):,}")

print(
    f"High similarity relationships: "
    f"{len(high_similarity_pairs):,}"
)

print(
    f"Average similarity score: "
    f"{upper_triangle.mean():.3f}"
)

print(
    f"Maximum similarity score: "
    f"{upper_triangle.max():.3f}"
)

print(
    f"Products with competitive similarity: "
    f"{competitive_opportunities['Product_ID'].nunique():,}"
)

print("\nStrategic categories:")

print(
    drug_intelligence[
        "Strategic_Similarity_Category"
    ].value_counts()
)

print("\nTop competitive opportunities:")

display(
    competitive_opportunities[
        [
            "Brand_Name",
            "Manufacturer",
            "Similar_Brand_Name",
            "Similar_Manufacturer",
            "Similarity_%",
            "Similar_Sales_Value"
        ]
    ].head(10)
)

print("\n" + "=" * 70)
print("DRUG SIMILARITY ANALYSIS COMPLETED")
print("=" * 70)

In [1]:
# ============================================================
# PharmaLens AI
# similarity.py
# Drug Similarity Engine
# ============================================================

import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import pairwise_distances


class DrugSimilarityEngine:

    def __init__(
        self,
        text_weight=0.35,
        categorical_weight=0.30,
        numerical_weight=0.15
    ):

        self.text_weight = text_weight
        self.categorical_weight = categorical_weight
        self.numerical_weight = numerical_weight

        self.tfidf = TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2)
        )

        self.encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )

        self.scaler = StandardScaler()

        self.product_df = None
        self.similarity_matrix = None

    @staticmethod
    def clean_text(value):

        if pd.isna(value):
            return ""

        value = str(value).lower()

        value = re.sub(
            r"[^a-zA-Z0-9\s]",
            " ",
            value
        )

        value = re.sub(
            r"\s+",
            " ",
            value
        )

        return value.strip()

    def prepare_products(self, data):

        text_columns = [
            "Distribution Channel",
            "Therapeutic Class",
            "Manufacturer",
            "Brand Name",
            "Pack Size",
            "Drug Strength",
            "Market Category"
        ]

        for col in text_columns:

            if col in data.columns:

                data[col] = (
                    data[col]
                    .fillna("Unknown")
                    .astype(str)
                    .str.strip()
                    .str.lower()
                )

        numeric_columns = [
            "Selling Price",
            "Sales Units",
            "Sales Value",
            "Year"
        ]

        for col in numeric_columns:

            if col in data.columns:

                data[col] = pd.to_numeric(
                    data[col],
                    errors="coerce"
                )

        data["Product Launch"] = pd.to_datetime(
            data["Product Launch"],
            errors="coerce"
        )

        product_columns = [
            "Brand Name",
            "Therapeutic Class",
            "Manufacturer",
            "Pack Size",
            "Drug Strength",
            "Market Category"
        ]

        products = (
            data
            .groupby(
                product_columns,
                dropna=False
            )
            .agg(
                Average_Price=(
                    "Selling Price",
                    "mean"
                ),

                Total_Sales_Units=(
                    "Sales Units",
                    "sum"
                ),

                Total_Sales_Value=(
                    "Sales Value",
                    "sum"
                ),

                First_Launch_Date=(
                    "Product Launch",
                    "min"
                ),

                Distribution_Channels=(
                    "Distribution Channel",
                    "nunique"
                )
            )
            .reset_index()
        )

        products.insert(
            0,
            "Product_ID",
            range(1, len(products) + 1)
        )

        products["Drug_Text_Profile"] = (
            products["Brand Name"].apply(
                self.clean_text
            )
            + " "
            + products["Therapeutic Class"].apply(
                self.clean_text
            )
            + " "
            + products["Manufacturer"].apply(
                self.clean_text
            )
            + " "
            + products["Pack Size"].apply(
                self.clean_text
            )
            + " "
            + products["Drug Strength"].apply(
                self.clean_text
            )
            + " "
            + products["Market Category"].apply(
                self.clean_text
            )
        )

        self.product_df = products

        return products

    def fit(self):

        df = self.product_df

        # -------------------------------
        # TEXT SIMILARITY
        # -------------------------------

        tfidf_matrix = self.tfidf.fit_transform(
            df["Drug_Text_Profile"]
        )

        text_similarity = cosine_similarity(
            tfidf_matrix
        )

        # -------------------------------
        # CATEGORICAL SIMILARITY
        # -------------------------------

        categorical_features = [
            "Therapeutic Class",
            "Manufacturer",
            "Market Category",
            "Drug Strength",
            "Pack Size"
        ]

        categorical_matrix = (
            self.encoder.fit_transform(
                df[categorical_features]
                .fillna("Unknown")
            )
        )

        categorical_similarity = cosine_similarity(
            categorical_matrix
        )

        # -------------------------------
        # NUMERICAL SIMILARITY
        # -------------------------------

        numerical_features = [
            "Average_Price",
            "Total_Sales_Units",
            "Total_Sales_Value",
            "Distribution_Channels"
        ]

        numerical_data = (
            df[numerical_features]
            .replace(
                [np.inf, -np.inf],
                np.nan
            )
        )

        numerical_data = numerical_data.fillna(
            numerical_data.median()
        )

        numerical_matrix = (
            self.scaler.fit_transform(
                numerical_data
            )
        )

        distance = pairwise_distances(
            numerical_matrix,
            metric="euclidean"
        )

        numerical_similarity = (
            1 / (1 + distance)
        )

        # -------------------------------
        # COMBINE
        # -------------------------------

        combined = (
            self.text_weight * text_similarity
            +
            self.categorical_weight *
            categorical_similarity
            +
            self.numerical_weight *
            numerical_similarity
        )

        combined = np.clip(
            combined,
            0,
            1
        )

        np.fill_diagonal(
            combined,
            1
        )

        self.similarity_matrix = combined

        return combined

    def find_similar(
        self,
        brand_name,
        top_n=10
    ):

        matches = self.product_df[
            self.product_df["Brand Name"]
            .str.lower()
            .str.strip()
            ==
            str(brand_name)
            .lower()
            .strip()
        ]

        if matches.empty:

            return pd.DataFrame()

        idx = matches.index[0]

        scores = self.similarity_matrix[idx]

        result = self.product_df.copy()

        result["Similarity_Score"] = scores

        result = result[
            result.index != idx
        ]

        result = (
            result
            .sort_values(
                "Similarity_Score",
                ascending=False
            )
            .head(top_n)
        )

        result["Similarity_%"] = (
            result["Similarity_Score"]
            * 100
        ).round(2)

        return result.reset_index(
            drop=True
        )

PharmaLens AI
│
├── notebooks/
│   ├── 01_Market_Intelligence.ipynb       ✅
│   ├── 02_Forecasting.ipynb
│   ├── 03_Molecule_Intelligence.ipynb     ✅
│   ├── 04_Launch_Success.ipynb             ✅
│   ├── 05_GTM_Intelligence.ipynb           ✅
│   ├── 06_Recommendation.ipynb             ✅
│   ├── 07_Company_Intelligence.ipynb
│   └── 08_Drug_Similarity.ipynb             ← THIS
│
├── src/
│   ├── market_intelligence.py
│   ├── forecasting.py
│   ├── molecule.py
│   ├── launch.py
│   ├── gtm.py
│   ├── recommendation.py
│   ├── company.py
│   └── similarity.py                         ← THIS
│
├── agent/
│   ├── tools.py
│   ├── agent.py
│   └── prompts.py
│
├── app/
│   └── streamlit_app.py
│
├── data/
│   └── drug_similarity_outputs/
│
├── models/
│   ├── drug_similarity_tfidf.pkl
│   ├── drug_similarity_encoder.pkl
│   ├── drug_similarity_scaler.pkl
│   └── drug_similarity_matrix.pkl
│
└── requirements.txt